# Clustering Energy Analysis

Thin runner notebook for LRK/xKV clustering comparisons across keys and values, with independent cells for each tensor and clustering axis.

In [1]:
from pathlib import Path
import sys

import torch


def find_repo_root(start: Path) -> Path:
    for path in (start, *start.parents):
        if (path / "utils").is_dir():
            return path
    raise RuntimeError("Could not find repo root containing ./utils")


repo_root = find_repo_root(Path.cwd().resolve())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from analysis.utils import display_case_result, load_kv_dump, run_analysis_case


/home/m84366023/miniconda3/envs/pem-llm/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
kv_dump_path = (
    repo_root
    / "results/kv_dumps/meta-llama_Llama-3.1-8B-Instruct/niah_multiquery_raw_kv.pt"
)
loaded = load_kv_dump(kv_dump_path)

raw_keys = loaded["raw_keys"]
keys = loaded["keys"]
values = loaded["values"]
prompt_len = loaded["prompt_len"]
rope_theta = loaded["rope_theta"]

keys.shape, values.shape, prompt_len, rope_theta


(torch.Size([32, 1, 8, 3770, 128]),
 torch.Size([32, 1, 8, 3770, 128]),
 3770,
 500000.0)

In [12]:
show_detail_tables = False

kmeans_cfg = {
    "n_clusters": 8,
    "kmeans_cluster_size": 2048,
    "kmeans_n_iter": 8,
    "kmeans_init": "infllm",
    "kmeans_dtype": torch.float32,
}

decomposition_cfg = {
    "decomposition_method": "svd",
    "rank_selection": "comp_ratio",
    "comp_ratio": 2.0,
    "energy_threshold": 0.95,
    "decomp_n_iter": 3,
    "decomp_lr": 1e-2,
}

lrk_mode = "per_head"  # "avg_heads" or "per_head"
include_lrk_head_breakdown = False
prefix_end = prompt_len
local_window = 0

xkv_layer_group_size = 4
xkv_num_layers = keys.size(0)


## Keys, Row Clustering

In [13]:
keys_rows = run_analysis_case(
    keys,
    tensor_name="keys",
    cluster_axis="rows",
    kmeans_cfg=kmeans_cfg,
    decomposition_cfg=decomposition_cfg,
    lrk_mode=lrk_mode,
    include_lrk_head_breakdown=include_lrk_head_breakdown,
    prefix_end=prefix_end,
    local_window=local_window,
    xkv_layer_group_size=xkv_layer_group_size,
    xkv_num_layers=xkv_num_layers,
)

display_case_result(keys_rows, show_detail_tables=show_detail_tables)


,case,rows,mean_eta,median_eta,mean_relative_low_rank_recon_error,median_relative_low_rank_recon_error,min_eta,max_eta
0,keys_lrk_rows,256,0.281595,0.266505,0.141856,0.148294,0.074367,0.521531
1,keys_xkv_rows,40,0.280649,0.279050,0.074527,0.077723,0.173713,0.434211
2,keys_lrk_rows_n_clusters_1,256,0.315100,0.304945,0.146556,0.153671,0.074367,0.558442
3,keys_xkv_rows_n_clusters_1,40,0.312275,0.306936,0.070334,0.072464,0.173713,0.473684


## Values, Row Clustering

In [14]:
values_rows = run_analysis_case(
    values,
    tensor_name="values",
    cluster_axis="rows",
    kmeans_cfg=kmeans_cfg,
    decomposition_cfg=decomposition_cfg,
    lrk_mode=lrk_mode,
    include_lrk_head_breakdown=include_lrk_head_breakdown,
    prefix_end=prefix_end,
    local_window=local_window,
    xkv_layer_group_size=xkv_layer_group_size,
    xkv_num_layers=xkv_num_layers,
)

display_case_result(values_rows, show_detail_tables=show_detail_tables)


,case,rows,mean_eta,median_eta,mean_relative_low_rank_recon_error,median_relative_low_rank_recon_error,min_eta,max_eta
0,values_lrk_rows,256,0.861804,0.897220,0.438724,0.443552,0.176020,0.982143
1,values_xkv_rows,40,0.869498,0.887734,0.233175,0.231238,0.505102,0.934066
2,values_lrk_rows_n_clusters_1,256,0.890863,0.923303,0.452579,0.458740,0.213010,0.989011
3,values_xkv_rows_n_clusters_1,40,0.884039,0.903943,0.217436,0.213246,0.528061,0.939560


## Keys, Column Clustering

In [10]:
keys_cols = run_analysis_case(
    keys,
    tensor_name="keys",
    cluster_axis="cols",
    kmeans_cfg=kmeans_cfg,
    decomposition_cfg=decomposition_cfg,
    lrk_mode=lrk_mode,
    include_lrk_head_breakdown=include_lrk_head_breakdown,
    prefix_end=prefix_end,
    local_window=local_window,
    xkv_layer_group_size=xkv_layer_group_size,
    xkv_num_layers=xkv_num_layers,
)

display_case_result(keys_cols, show_detail_tables=show_detail_tables)


,case,rows,mean_eta,median_eta,mean_relative_low_rank_recon_error,median_relative_low_rank_recon_error,min_eta,max_eta
0,keys_lrk_cols,256,0.992535,0.993506,0.146559,0.153671,0.932039,1.000000
1,keys_xkv_cols,40,0.480275,0.489612,0.087195,0.090008,0.283088,0.578947
2,keys_lrk_cols_n_clusters_1,256,0.992535,0.993506,0.146559,0.153671,0.932039,1.000000
3,keys_xkv_cols_n_clusters_1,40,0.998950,1.000000,0.070360,0.072464,0.992481,1.000000


## Values, Column Clustering

In [11]:
values_cols = run_analysis_case(
    values,
    tensor_name="values",
    cluster_axis="cols",
    kmeans_cfg=kmeans_cfg,
    decomposition_cfg=decomposition_cfg,
    lrk_mode=lrk_mode,
    include_lrk_head_breakdown=include_lrk_head_breakdown,
    prefix_end=prefix_end,
    local_window=local_window,
    xkv_layer_group_size=xkv_layer_group_size,
    xkv_num_layers=xkv_num_layers,
)

display_case_result(values_cols, show_detail_tables=show_detail_tables)


,case,rows,mean_eta,median_eta,mean_relative_low_rank_recon_error,median_relative_low_rank_recon_error,min_eta,max_eta
0,values_lrk_cols,256,0.991860,0.992308,0.452579,0.458740,0.962963,1.000000
1,values_xkv_cols,40,0.916953,0.930971,0.260621,0.261833,0.612245,0.983425
2,values_lrk_cols_n_clusters_1,256,0.991860,0.992308,0.452579,0.458740,0.962963,1.000000
3,values_xkv_cols_n_clusters_1,40,0.999150,1.000000,0.217347,0.213246,0.993827,1.000000
